# pyDBSI — Simulazioni di accuratezza (toolbox ≥ 1.8.1)

**Domanda.** Quanto le mappe del toolbox si avvicinano alla verità? Sui dati veri non c'è una ground
truth (asse 3 del disegno dei 5P: l'accuratezza si misura **solo** in simulazione). Qui la verità è nota
voxel per voxel.

**La catena è quella di produzione**, non una scorciatoia:

> segnale vero (S0 = 1000) → rumore **Rician** (σ = S0/SNR) → **`dwidenoise`** (MRtrix, MP-PCA) →
> mappa `-noise` di dwidenoise come `noise_map` → `DBSI_Adaptive().fit(run_calibration=True)`

cioè quello che succede ai 5P: calibrazione libera, mappa di rumore di dwidenoise. Non c'è N4 (il fantoccio
non ha campo di bias).

**Due bracci, e perché il principale è il grezzo.** Il braccio **grezzo** salta il denoising e passa la σ
vera; il braccio **produzione** passa da dwidenoise. Sul fantoccio MP-PCA è troppo efficace: toglie il
92–96% del rumore (il fantoccio è a basso rango, il tessuto vero no), e la calibrazione libera scende a
λ_aniso ≈ 0.5 contro 8.4 del grezzo e 5–13 dei dati veri, che **sono** denoisati (prova del 2026-10-12, P3,
SNR 26). Il braccio grezzo riproduce il regime di λ dei dati veri, quindi è quello delle figure principali; il
braccio con dwidenoise gira a `SNR_PRODUZIONE` come **analisi di sensibilità su un caso idealizzato**: dice
cosa farebbe il fit se il denoising fosse così efficace, non cosa fa sui dati veri.

**Un volume per condizione.** I tre fantocci stanno affiancati in un unico volume, così ogni condizione
(protocollo × SNR × braccio) ha **una** calibrazione e **un** dwidenoise, come un cervello con molti tessuti:

| fantoccio | che cosa mostra | figure |
|---|---|---|
| **anatomico** | fetta con GM, due fasci che si incrociano a 90° e un terzo a 60°, ventricolo, lesioni demielinizzante / infiammatoria / edematosa, tumore; volume parziale ai bordi | immagini vero / stimato / errore |
| **campi** | tutti i parametri variano con continuità (campi gaussiani lisci), metà voxel con incrocio 45–90° | stima contro verità, voxel per voxel |
| **batteria** | i 13 scenari della validazione di luglio, blocchi omogenei con eterogeneità lieve | errore per scenario, per SNR e per braccio; conteggio delle popolazioni; incertezza |

**Diffusività isotrope vere ≠ centroidi di Stage D.** Con `D_VARIABILE = True` (default) le diffusività
vere sono estratte per voxel (R 0.10–0.25, H 0.7–1.3, W 2.8–3.2 ×1e-3 mm²/s), mentre Stage D usa centroidi
fissi 0.15 / 1.0 / 3.0: è il caso reale, il modello non è esatto. Con `False` si misura lo stimatore quando
il modello è esatto.

**Quanto dura.** La calibrazione libera costa ~20–25 min per condizione (quasi tutto è il primo passaggio di
discrepanza su λ_aniso: NNLS su 1000 voxel a λ quasi nullo). Default: 3 condizioni grezze sul P3 + 1 con
dwidenoise + 1 per ogni altro protocollo trovato. La condizione con dwidenoise è la più lenta (a λ_aniso ≈ 0.5
la NNLS di Stage A arriva al tetto di iterazioni in quasi ogni voxel): togliere `'produzione'` da `BRACCI` per
saltarla. In tutto **3–5 ore**. Ogni condizione
è salvata appena finita in `risultati_simulazioni/v<versione>/`: se il notebook si interrompe, rilanciandolo riparte
da dove era. `RAPIDO = True` fa una prova con fantocci dimezzati (numeri solo indicativi).

**Dove lanciarlo.** Sul fisso, dove ci sono `dwidenoise` e le tabelle dei 5 protocolli sul NAS (senza NAS
gira il solo P3, la cui tabella è dentro il notebook). Da una cartella qualsiasi:
`jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.timeout=-1 simulazioni_pyDBSI.ipynb`
oppure dal browser, *Run All*.

## 0 — Configurazione

In [ ]:
RAPIDO      = False                 # True: fantocci dimezzati, prova veloce (numeri indicativi)
RIFARE      = False                 # True: ignora le condizioni gia' salvate
D_VARIABILE = True                  # diffusivita' isotrope vere estratte per voxel (vedi sopra)
SNR_LISTA   = [15, 26, 40]          # SNR termico (S0 / sigma) delle condizioni sul P3
SNR_RIF     = 26                    # SNR delle figure principali e del confronto fra protocolli
BRACCI      = ['grezzo', 'produzione']  # il primo e' quello delle figure principali (vedi sopra)
SNR_PRODUZIONE = [26]               # SNR a cui gira anche il braccio con dwidenoise (sensibilita')
DWIDENOISE  = None                  # None = cercato da solo (PATH, /usr/local/mrtrix3); oppure il percorso
N_CAL       = 1000                  # voxel di calibrazione (default del toolbox, come nei fit veri)

# Tabelle dei 5P sul NAS: se ci sono, la batteria gira anche su P1, P2, P4, P5 (a SNR_RIF, produzione).
ROOT_5P = "/mnt/msbiobnas/Users/Francesco/DBSI_FG/DATA_5P/Processed"
STEM    = "DWITE106P2sense19nsoppyAPP"
SESSIONE = "20251210"

## 1 — Il codice delle simulazioni

Due celle lunghe: i fantocci, la catena e le misure; poi le figure. Non serve leggerle per usare il
notebook. (Sorgenti in `notebooks/sorgenti/`: `sim_accuratezza.py`, `sim_grafici.py`; il notebook si rigenera con
`python notebooks/sorgenti/build_simulazioni.py`.)

In [ ]:
"""
Simulazioni di accuratezza per pyDBSI (>= 1.8.1): quanto le mappe del toolbox si avvicinano alla verita'.

Tutto passa per la CATENA DI PRODUZIONE, non per una scorciatoia:

    segnale vero (S0 = 1000)  ->  rumore Rician (sigma = S0 / SNR)  ->  dwidenoise (MRtrix, MP-PCA)
        ->  mappa -noise di dwidenoise come noise_map  ->  DBSI_Adaptive().fit(run_calibration=True)

cioe' cio' che succede ai 5P (calibrazione libera, mappa di rumore di dwidenoise). Il braccio "grezzo"
salta il denoising e passa la sigma vera: misura lo stimatore da solo, e la differenza fra i due bracci
e' l'effetto del denoising. Non c'e' N4: il fantoccio non ha campo di bias.

Tre fantocci, tre domande:
  - BATTERIA     13 scenari omogenei (quelli della validazione di luglio), blocchi 3D con eterogeneita'
                 lieve: per ogni scenario, bias e dispersione di ogni mappa.
  - ANATOMICO    una fetta (48x48x6) "a forma di cervello" (GM, due fasci che si incrociano a 90 gradi, un terzo a
                 60, ventricolo, lesioni demielinizzante / edematosa / infiammatoria, tumore) con volume
                 parziale ai bordi: le immagini vero / stimato / errore.
  - CAMPI        parametri che variano con continuita' nello spazio (campi gaussiani lisci): stima contro
                 verita' voxel per voxel, su tutto l'intervallo dei valori.

La verita' delle diffusivita' isotrope NON coincide coi centroidi fissi di Stage D (0.15 / 1.0 / 3.0e-3)
se D_VARIABILE: R in [0.10, 0.25], H in [0.7, 1.3], W in [2.8, 3.2] e-3 mm^2/s, estratte per voxel. E' il
caso reale (il tessuto non ha diffusivita' esattamente uguali ai centroidi); con D_VARIABILE=False si
misura lo stimatore nel caso in cui il modello e' esatto.
"""

import contextlib
import io
import os
import shutil
import subprocess
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

import dbsi_toolbox
from dbsi_toolbox import DBSI_Adaptive

S0 = 1000.0
D_NOM = dict(R=0.15e-3, H=1.0e-3, W=3.0e-3)
D_RANGE = dict(R=(0.10e-3, 0.25e-3), H=(0.7e-3, 1.3e-3), W=(2.8e-3, 3.2e-3))

# metriche riportate: nome breve -> canale del toolbox
METRICHE = dict(FF='fiber_fraction', RF='restricted_fraction', HF='hindered_fraction',
                WF='water_fraction', ADw='axial_diffusivity_weighted',
                RDw='radial_diffusivity_weighted', FAw='fiber_fa_weighted')
FRAZIONI = ('FF', 'RF', 'HF', 'WF')
TENSORE = ('ADw', 'RDw', 'FAw')
UNITA = dict(FF='', RF='', HF='', WF='', ADw='e-3 mm²/s', RDw='e-3 mm²/s', FAw='')
SCALA = dict(FF=1, RF=1, HF=1, WF=1, ADw=1e3, RDw=1e3, FAw=1)    # diffusivita' mostrate in e-3

NOMI = DBSI_Adaptive.output_map_names(3)
IX = {n: i for i, n in enumerate(NOMI)}


# ─────────────────────────────────────────────────────────────────────────────
# protocollo
# ─────────────────────────────────────────────────────────────────────────────
def carica_protocollo(bval, bvec):
    """bval/bvec in formato FSL; bvec normalizzati, righe (N, 3)."""
    bv = np.loadtxt(bval).ravel().astype(float)
    bc = np.loadtxt(bvec)
    bc = bc.T if bc.shape[0] == 3 and bc.shape[1] != 3 else bc
    n = np.linalg.norm(bc, axis=1, keepdims=True)
    n[n == 0] = 1.0
    return bv, bc / n


def descrivi_protocollo(prot):
    bv = prot[0]
    gusci = pd.Series(np.round(bv, -2).astype(int)).value_counts().sort_index()
    return ', '.join(f"b={b}: {n}" for b, n in gusci.items())


# ─────────────────────────────────────────────────────────────────────────────
# tessuto -> segnale e verita'
# ─────────────────────────────────────────────────────────────────────────────
def direzione(theta_deg, phi_deg):
    t, p = np.radians(theta_deg), np.radians(phi_deg)
    v = np.array([np.sin(t) * np.cos(p), np.sin(t) * np.sin(p), np.cos(t)])
    return v / np.linalg.norm(v)


def ruota_verso(u, gradi, asse_aux=(0.0, 0.0, 1.0)):
    """Direzione a `gradi` gradi da u, nel piano di u e di asse_aux."""
    a = np.asarray(asse_aux, float)
    t = a - (a @ u) * u
    if np.linalg.norm(t) < 1e-6:
        t = np.cross(u, [1.0, 0, 0])
    t /= np.linalg.norm(t)
    g = np.radians(gradi)
    return np.cos(g) * u + np.sin(g) * t


def segnale(prot, fibre, iso, D):
    """fibre: [(f, dir, AD, RD)], iso: {'R': f, 'H': f, 'W': f}, D: {'R': D, ...}. S(b=0) = somma f."""
    bv, bc = prot
    S = np.zeros(len(bv))
    for k, f in iso.items():
        if f > 0:
            S += f * np.exp(-bv * D[k])
    for f, d, ad, rd in fibre:
        if f > 0:
            S += f * np.exp(-bv * (rd + (ad - rd) * (bc @ d) ** 2))
    return S


def fa(ad, rd):
    return np.abs(ad - rd) / np.sqrt(ad ** 2 + 2 * rd ** 2)


SOGLIA_POP = 0.05     # una popolazione esiste, per la verita', se pesa almeno il 5% del voxel


def verita(fibre, iso, soglia_pop=SOGLIA_POP):
    """Valori veri delle metriche riportate (frazioni normalizzate al totale). Il tensore vero e' definito
    solo dalle popolazioni sopra soglia: una coda di volume parziale da 1% non ha un tensore misurabile."""
    tot = sum(f for f, *_ in fibre) + sum(iso.values())
    ff = sum(f for f, *_ in fibre)
    vive = [x for x in fibre if x[0] / tot >= soglia_pop]
    t = dict(FF=ff / tot, RF=iso.get('R', 0) / tot, HF=iso.get('H', 0) / tot, WF=iso.get('W', 0) / tot,
             npop=len(vive))
    fv = sum(f for f, *_ in vive)
    if fv > 0:
        adw = sum(f * ad for f, _, ad, _ in vive) / fv
        rdw = sum(f * rd for f, _, _, rd in vive) / fv
        t.update(ADw=adw, RDw=rdw, FAw=fa(adw, rdw))
    else:
        t.update(ADw=np.nan, RDw=np.nan, FAw=np.nan)
    return t


def estrai_D(rng, variabile):
    if not variabile:
        return dict(D_NOM)
    return {k: rng.uniform(*D_RANGE[k]) for k in D_NOM}


# ─────────────────────────────────────────────────────────────────────────────
# BATTERIA: 13 scenari (validazione di luglio), blocchi 3D omogenei
# ─────────────────────────────────────────────────────────────────────────────
_d0, _d1, _d2 = direzione(90, 0), direzione(65, 0), direzione(65, 90)


def _I(r, h, w):
    return dict(R=r, H=h, W=w)


SCENARI = {
    'WM sana':          ([(0.55, _d0, 1.7e-3, 0.30e-3)], _I(0.10, 0.25, 0.10)),
    'WM demielinizzata': ([(0.45, _d0, 1.5e-3, 0.70e-3)], _I(0.15, 0.25, 0.15)),
    'WM densa':         ([(0.70, _d0, 1.8e-3, 0.25e-3)], _I(0.10, 0.15, 0.05)),
    'WM debole':        ([(0.20, _d0, 1.7e-3, 0.30e-3)], _I(0.10, 0.50, 0.20)),
    'Crossing bil.':    ([(0.35, _d1, 1.6e-3, 0.40e-3), (0.35, _d2, 1.6e-3, 0.40e-3)], _I(0.10, 0.15, 0.05)),
    'Crossing sbil.':   ([(0.45, _d1, 1.7e-3, 0.30e-3), (0.20, _d2, 1.6e-3, 0.40e-3)], _I(0.10, 0.15, 0.10)),
    'Crossing eterog.': ([(0.30, _d1, 1.7e-3, 0.30e-3), (0.20, _d2, 1.4e-3, 0.65e-3)], _I(0.18, 0.14, 0.18)),
    'GM':               ([(0.10, _d0, 1.5e-3, 0.50e-3)], _I(0.10, 0.65, 0.15)),
    'Liquor':           ([], _I(0.0, 0.05, 0.95)),
    'Solo restricted':  ([], _I(0.30, 0.50, 0.20)),
    'Tumore':           ([], _I(0.55, 0.30, 0.15)),
    'Edema':            ([(0.30, _d0, 1.7e-3, 0.35e-3)], _I(0.0, 0.15, 0.55)),
    'Lesione infiamm.': ([(0.25, _d0, 1.6e-3, 0.45e-3)], _I(0.35, 0.30, 0.10)),
}


def fantoccio_batteria(prot, blocco=(8, 8, 6), colonne=7, eterog=0.05, d_variabile=True, seed=0, scenari=None):
    """Un blocco 3D per scenario, su una griglia di `colonne` blocchi. Eterogeneita' lieve nel blocco:
    frazioni x(1 + N(0, eterog)), direzioni che ruotano di qualche grado con la posizione, D isotrope per
    voxel. Ritorna (A pulito in unita' di segnale, verita' {metrica: (X,Y,Z)}, etichette (X,Y,Z) int con -1
    fuori dai blocchi, nomi)."""
    scenari = scenari or SCENARI
    rng = np.random.default_rng(seed)
    bx, by, bz = blocco
    nomi = list(scenari)
    righe = int(np.ceil(len(nomi) / colonne))
    X, Y, N = bx * colonne, by * righe, len(prot[0])
    A = np.zeros((X, Y, bz, N), np.float32)
    lab = np.full((X, Y, bz), -1, np.int16)
    tru = {k: np.full((X, Y, bz), np.nan) for k in (*METRICHE, 'npop')}
    tru['dirs'] = np.full((X, Y, bz, 2, 3), np.nan)
    for s, nome in enumerate(nomi):
        x0, y0 = (s % colonne) * bx, (s // colonne) * by
        fibre0, iso0 = scenari[nome]
        for i in range(bx):
            for j in range(by):
                for k in range(bz):
                    jit = 1 + rng.normal(0, eterog, 3 + len(fibre0))
                    iso = {c: iso0[c] * jit[q] for q, c in enumerate('RHW')}
                    fibre = []
                    for q, (f, d, ad, rd) in enumerate(fibre0):
                        dd = ruota_verso(d, 4.0 * (i - bx / 2) / bx + 3.0 * (j - by / 2) / by)
                        fibre.append((f * jit[3 + q], dd, ad, rd))
                    D = estrai_D(rng, d_variabile)
                    S = segnale(prot, fibre, iso, D)
                    tot = sum(f for f, *_ in fibre) + sum(iso.values())
                    x, y = x0 + i, y0 + j
                    A[x, y, k] = S / tot
                    lab[x, y, k] = s
                    for m, v in verita(fibre, iso).items():
                        tru[m][x, y, k] = v
                    for q, (_, dd, _, _) in enumerate(fibre[:2]):
                        tru['dirs'][x, y, k, q] = dd
    return A, tru, lab, nomi


# ─────────────────────────────────────────────────────────────────────────────
# ANATOMICO: una fetta con fasci, incroci, ventricolo e lesioni, volume parziale ai bordi
# ─────────────────────────────────────────────────────────────────────────────
def _morbida(dist, larghezza=0.8):
    """Appartenenza in [0, 1] da una distanza con segno (negativa dentro)."""
    return 1.0 / (1.0 + np.exp(dist / larghezza))


TESSUTI_ANAT = {
    # nome: (fibre come [(f, chiave_direzione, AD, RD)], iso)
    'GM':              ([], _I(0.10, 0.75, 0.15)),
    'fascio A':        ([(0.60, 'A', 1.7e-3, 0.30e-3)], _I(0.10, 0.22, 0.08)),
    'fascio B':        ([(0.60, 'B', 1.7e-3, 0.30e-3)], _I(0.10, 0.22, 0.08)),
    'fascio C':        ([(0.60, 'C', 1.7e-3, 0.35e-3)], _I(0.10, 0.22, 0.08)),
    'liquor':          ([], _I(0.0, 0.05, 0.95)),
    'demielinizzata':  ([(0.45, 'A', 1.5e-3, 0.75e-3)], _I(0.10, 0.35, 0.10)),
    'edema':           ([(0.30, 'B', 1.7e-3, 0.35e-3)], _I(0.05, 0.20, 0.45)),
    'infiammazione':   ([(0.40, 'A', 1.65e-3, 0.40e-3)], _I(0.30, 0.22, 0.08)),
    'tumore':          ([], _I(0.50, 0.35, 0.15)),
}


def fantoccio_anatomico(prot, nx=64, ny=64, nz=6, d_variabile=True, seed=1):
    """Ritorna (A, verita', pesi {tessuto: (X,Y,Z)}, direzioni vere per voxel {chiave: (X,Y,Z,3)})."""
    rng = np.random.default_rng(seed)
    x, y = np.meshgrid(np.arange(nx) + 0.5, np.arange(ny) + 0.5, indexing='ij')
    c = np.array([nx / 2, ny / 2])
    # fascio A: banda orizzontale leggermente curva; B: banda verticale (incrocio a 90 con A);
    # C: banda obliqua a 60 gradi rispetto ad A, nella meta' bassa
    yA = ny * 0.62 + 4 * np.sin(2 * np.pi * x / nx)
    m = {}
    m['fascio A'] = _morbida(np.abs(y - yA) - 6)
    m['fascio B'] = _morbida(np.abs(x - nx * 0.30) - 5)
    ang = np.radians(60)
    dC = np.abs((x - nx * 0.70) * np.sin(ang) - (y - ny * 0.25) * np.cos(ang))
    m['fascio C'] = _morbida(dC - 4.5) * _morbida(y - ny * 0.50, 1.5)
    m['liquor'] = _morbida(np.hypot((x - nx * 0.55) / 7, (y - ny * 0.40) / 4) * 4 - 4)
    def disco(cx, cy, r):
        return _morbida(np.hypot(x - cx, y - cy) - r)
    m['demielinizzata'] = disco(nx * 0.80, ny * 0.62 + 4 * np.sin(2 * np.pi * 0.80), 4.0)    # sul fascio A
    m['infiammazione'] = disco(nx * 0.55, ny * 0.62 + 4 * np.sin(2 * np.pi * 0.55), 3.5)     # sul fascio A
    m['edema'] = disco(nx * 0.30, ny * 0.15, 4.0)                                          # sul fascio B
    m['tumore'] = disco(nx * 0.15, ny * 0.40, 4.5)
    # composizione dall'alto (alpha compositing): liquor > lesioni > fasci > GM. I fasci si dividono
    # la copertura dove si sovrappongono: e' li' che nasce l'incrocio.
    w = {k: np.zeros((nx, ny)) for k in TESSUTI_ANAT}
    resto = np.ones((nx, ny))
    for k in ('liquor', 'tumore', 'demielinizzata', 'infiammazione', 'edema'):
        w[k] = resto * m[k]
        resto = resto - w[k]
    fasci = ('fascio A', 'fascio B', 'fascio C')
    somma = sum(m[k] for k in fasci)
    cop = np.minimum(somma, 1.0)
    for k in fasci:
        w[k] = resto * cop * np.where(somma > 0, m[k] / np.maximum(somma, 1e-12), 0)
    w['GM'] = resto * (1 - cop)
    # direzioni: A segue la curva, B verticale, C a 60 gradi da x; inclinazione fuori piano 15 gradi
    dirs = {}
    slopeA = 4 * 2 * np.pi / nx * np.cos(2 * np.pi * x / nx)
    a = np.arctan(slopeA)
    el = np.radians(15)
    dirs['A'] = np.stack([np.cos(a) * np.cos(el), np.sin(a) * np.cos(el), np.full_like(a, np.sin(el))], -1)
    dirs['B'] = np.broadcast_to(direzione(90 - 15, 90), (nx, ny, 3)).copy()
    dirs['C'] = np.broadcast_to(direzione(90 - 15, 60), (nx, ny, 3)).copy()
    N = len(prot[0])
    A = np.zeros((nx, ny, nz, N), np.float32)
    tru = {k: np.full((nx, ny, nz), np.nan) for k in (*METRICHE, 'npop')}
    for i in range(nx):
        for j in range(ny):
            for k in range(nz):
                D = estrai_D(rng, d_variabile)
                fib_tot, iso_tot = {}, dict(R=0.0, H=0.0, W=0.0)
                for t, wt in w.items():
                    ww = wt[i, j]
                    if ww < 1e-4:
                        continue
                    fl, iso = TESSUTI_ANAT[t]
                    jit = 1 + rng.normal(0, 0.04, 3)
                    for q, cc in enumerate('RHW'):
                        iso_tot[cc] += ww * iso[cc] * jit[q]
                    for f, key, ad, rd in fl:
                        a0 = fib_tot.get(key, [0.0, 0.0, 0.0])
                        fib_tot[key] = [a0[0] + ww * f, a0[1] + ww * f * ad, a0[2] + ww * f * rd]
                fibre = [(v[0], dirs[key][i, j], v[1] / v[0], v[2] / v[0]) for key, v in fib_tot.items()
                         if v[0] > 1e-4]
                S = segnale(prot, fibre, iso_tot, D)
                tot = sum(f for f, *_ in fibre) + sum(iso_tot.values())
                A[i, j, k] = S / tot
                for mm, v in verita(fibre, iso_tot).items():
                    tru[mm][i, j, k] = v
    pesi = {t: np.repeat(wt[:, :, None], nz, 2) for t, wt in w.items()}
    return A, tru, pesi


# ─────────────────────────────────────────────────────────────────────────────
# CAMPI: parametri lisci e continui
# ─────────────────────────────────────────────────────────────────────────────
def _campo(rng, shape, scala):
    from scipy.ndimage import gaussian_filter
    g = gaussian_filter(rng.normal(size=shape), scala, mode='wrap')
    g = (g - g.mean()) / (g.std() + 1e-12)
    from scipy.stats import norm
    return norm.cdf(g)              # uniforme in (0, 1), liscio


def fantoccio_campi(prot, shape=(48, 48, 8), scala=3.0, d_variabile=True, seed=2):
    """FF, frazioni iso, AD, RD, angolo di incrocio e quota della seconda fibra variano con continuita'.
    Meta' del volume ha due fibre (incrocio con angolo fra 45 e 90 gradi), meta' una."""
    rng = np.random.default_rng(seed)
    u = {k: _campo(rng, shape, scala) for k in ('ff', 'r', 'h', 'w', 'ad', 'rd', 'th', 'ph', 'ang', 'q2', 'cross')}
    N = len(prot[0])
    A = np.zeros(shape + (N,), np.float32)
    tru = {k: np.full(shape, np.nan) for k in (*METRICHE, 'npop')}
    for idx in np.ndindex(shape):
        g = {k: v[idx] for k, v in u.items()}
        ff = 0.75 * g['ff']                                   # 0 - 0.75
        wiso = np.array([0.05 + g['r'], 0.05 + 1.5 * g['h'], 0.02 + 0.8 * g['w']])
        wiso = wiso / wiso.sum() * (1 - ff)
        ad = 1.3e-3 + 0.7e-3 * g['ad']
        rd = min(0.15e-3 + 0.65e-3 * g['rd'], ad / 1.6)
        d1 = direzione(30 + 120 * g['th'], 360 * g['ph'])
        fibre = []
        if ff > 0.03:
            if g['cross'] > 0.5:
                d2 = ruota_verso(d1, 45 + 45 * g['ang'], asse_aux=direzione(90, 360 * g['ph'] + 90))
                q2 = 0.3 + 0.4 * g['q2']
                fibre = [(ff * (1 - q2), d1, ad, rd), (ff * q2, d2, ad, rd)]
            else:
                fibre = [(ff, d1, ad, rd)]
        iso = dict(R=wiso[0], H=wiso[1], W=wiso[2])
        D = estrai_D(rng, d_variabile)
        S = segnale(prot, fibre, iso, D)
        A[idx] = S / (sum(f for f, *_ in fibre) + wiso.sum())
        for m, v in verita(fibre, iso).items():
            tru[m][idx] = v
    return A, tru


# ─────────────────────────────────────────────────────────────────────────────
# VOLUME UNICO: i tre fantocci affiancati, una calibrazione e un dwidenoise per condizione
# ─────────────────────────────────────────────────────────────────────────────
def volume_composito(prot, d_variabile=True, seed=0, piccolo=False, spazio=4):
    """Anatomico | campi | batteria affiancati lungo x, separati da `spazio` voxel di fondo (segnale
    nullo, fuori maschera). La calibrazione campiona i voxel di tutti e tre, come in un cervello con molti
    tessuti; dwidenoise vede un volume unico. piccolo=True dimezza i fantocci (prova veloce).
    Ritorna (A, maschera, parti) con parti[nome] = dict(sl=slice lungo x, verita'=..., ...)."""
    if piccolo:
        an = fantoccio_anatomico(prot, nx=40, ny=40, nz=6, d_variabile=d_variabile, seed=seed + 1)
        ca = fantoccio_campi(prot, shape=(28, 28, 6), d_variabile=d_variabile, seed=seed + 2)
        ba = fantoccio_batteria(prot, blocco=(6, 6, 6), colonne=7, d_variabile=d_variabile, seed=seed)
    else:
        an = fantoccio_anatomico(prot, nx=48, ny=48, nz=6, d_variabile=d_variabile, seed=seed + 1)
        ca = fantoccio_campi(prot, shape=(32, 32, 6), d_variabile=d_variabile, seed=seed + 2)
        ba = fantoccio_batteria(prot, blocco=(8, 8, 6), colonne=7, d_variabile=d_variabile, seed=seed)
    pezzi = [('anatomico', an[0]), ('campi', ca[0]), ('batteria', ba[0])]
    Y = max(p.shape[1] for _, p in pezzi)
    Z = pezzi[0][1].shape[2]
    X = sum(p.shape[0] for _, p in pezzi) + spazio * (len(pezzi) + 1)
    N = len(prot[0])
    A = np.zeros((X, Y, Z, N), np.float32)
    mask = np.zeros((X, Y, Z), bool)
    parti, x0 = {}, spazio
    for nome, P in pezzi:
        nx, ny = P.shape[:2]
        sl = (slice(x0, x0 + nx), slice(0, ny))
        A[sl] = P
        mask[sl] = P.max(-1) > 0          # il posto vuoto della griglia della batteria resta fuori
        parti[nome] = dict(sl=sl)
        x0 += nx + spazio
    parti['anatomico'].update(verita=an[1], pesi=an[2])
    parti['campi'].update(verita=ca[1])
    parti['batteria'].update(verita=ba[1], etichette=ba[2], nomi=ba[3])
    return A, mask, parti


def ritaglia(mappe, sl):
    """Le mappe stimate ({nome: (X,Y,Z)}) ritagliate su una parte del volume."""
    return {k: v[sl] for k, v in mappe.items()}


# ─────────────────────────────────────────────────────────────────────────────
# catena: rumore -> dwidenoise -> fit
# ─────────────────────────────────────────────────────────────────────────────
def rumore_rician(A, snr, seed=0):
    rng = np.random.default_rng(seed)
    sig = S0 / snr
    X = S0 * A.astype(np.float64)
    M = np.sqrt((X + rng.normal(0, sig, X.shape)) ** 2 + rng.normal(0, sig, X.shape) ** 2)
    return M.astype(np.float32), sig


def trova_dwidenoise(percorso=None):
    for p in (percorso, shutil.which('dwidenoise'), '/usr/local/mrtrix3/dwidenoise',
              '/usr/local/mrtrix3/bin/dwidenoise'):
        if p and os.path.isfile(p) and os.access(p, os.X_OK):
            return p
    return None


def denoise(M, eseguibile, voxel_mm=2.0):
    """dwidenoise di MRtrix sui dati (X,Y,Z,N): ritorna (denoisati, mappa -noise)."""
    import nibabel as nib
    aff = np.diag([voxel_mm, voxel_mm, voxel_mm, 1.0])
    with tempfile.TemporaryDirectory() as td:
        fi, fo, fn = (os.path.join(td, n) for n in ('in.nii', 'den.nii', 'noise.nii'))
        nib.save(nib.Nifti1Image(M.astype(np.float32), aff), fi)
        subprocess.run([eseguibile, fi, fo, '-noise', fn, '-force', '-quiet'], check=True)
        D = np.asarray(nib.load(fo).dataobj, dtype=np.float32)
        nm = np.asarray(nib.load(fn).dataobj, dtype=np.float64)
    return D, nm


def fit(data, prot, noise_map, mask=None, **kw_fit):
    """Fit di produzione (calibrazione libera salvo kw_fit): ritorna (risultati, modello)."""
    mask = np.ones(data.shape[:3], bool) if mask is None else mask
    m = DBSI_Adaptive()
    kw = dict(run_calibration=True)
    kw.update(kw_fit)
    with contextlib.redirect_stdout(io.StringIO()):
        res, mode = m.fit(data, prot[0], prot[1], mask, noise_map=noise_map, **kw)
    assert mode == 3, f'modo {mode}-ISO: il protocollo non regge il 3-ISO'
    return res, m


def esegui(A, prot, snr, braccio, dwidenoise=None, seed=0, mask=None, **kw_fit):
    """Un braccio della catena. braccio = 'produzione' (dwidenoise + sua mappa) o 'grezzo' (sigma vera).
    Il rumore e' aggiunto a TUTTO il volume, fondo compreso (come in uno scanner); il fit solo nella maschera."""
    M, sig = rumore_rician(A, snr, seed)
    if braccio == 'produzione':
        if dwidenoise is None:
            raise RuntimeError('dwidenoise non trovato: braccio di produzione impossibile')
        D, nm = denoise(M, dwidenoise)
        data, noise = D, nm
    elif braccio == 'grezzo':
        data, noise = M, np.full(A.shape[:3], sig)
    else:
        raise ValueError(braccio)
    res, m = fit(data, prot, noise, mask=mask, **kw_fit)
    dentro = np.ones(A.shape[:3], bool) if mask is None else mask
    info = dict(snr=snr, braccio=braccio, sigma_vera=sig, sigma_mappa_mediana=float(np.median(noise[dentro])),
                lambda_aniso=float(m.lambda_aniso), lambda_iso=float(m.lambda_iso), n_iso=int(m.n_iso))
    return res, m, info


def stima(res, m=None):
    """Mappe stimate col nome breve. Le frazioni sono definite in ogni voxel fittato; il tensore e' NaN
    dove il toolbox non ha trovato fibre (lasciato NaN: e' "fibra mancata", contato a parte)."""
    out = {k: res[..., IX[c]].astype(float) for k, c in METRICHE.items()}
    out['npop'] = np.nan_to_num(res[..., IX['n_fiber_populations']].astype(float))
    if m is not None and getattr(m, 'uncertainty_', None) is not None:
        se = m.uncertainty_['se']
        for k, c in METRICHE.items():
            out['SE_' + k] = se[..., IX[c]].astype(float)
    return out


# ─────────────────────────────────────────────────────────────────────────────
# misure
# ─────────────────────────────────────────────────────────────────────────────
def interno(lab, margine=1):
    """Maschera dei voxel ad almeno `margine` voxel dal bordo del proprio blocco (lungo x e y): ai
    bordi dwidenoise mescola blocchi diversi, come al confine fra tessuti."""
    ok = lab >= 0
    for ax in (0, 1):
        for s in range(1, margine + 1):
            ok &= np.roll(lab, s, ax) == lab
            ok &= np.roll(lab, -s, ax) == lab
    return ok


def tabella_errori(est, tru, gruppi, nomi_gruppi, sel=None):
    """Per gruppo e metrica: vero (mediana), stima (mediana), bias (media), MAE, sd, n validi, e per le
    metriche del tensore la quota di voxel con fibra vera in cui la stima manca (NaN)."""
    righe = []
    sel = np.ones(gruppi.shape, bool) if sel is None else sel
    for g, nome in enumerate(nomi_gruppi):
        s = (gruppi == g) & sel
        for k in METRICHE:
            t, e = tru[k][s], est[k][s]
            ok = np.isfinite(t) & np.isfinite(e)
            r = dict(gruppo=nome, metrica=k, n=int(s.sum()), n_validi=int(ok.sum()))
            if k in TENSORE:
                vf = np.isfinite(t)
                r['mancanti_pct'] = float(100 * np.mean(~np.isfinite(e[vf]))) if vf.any() else np.nan
            if ok.sum():
                d = (e[ok] - t[ok]) * SCALA[k]
                r.update(vero=float(np.median(t[ok]) * SCALA[k]), stima=float(np.median(e[ok]) * SCALA[k]),
                         bias=float(d.mean()), mae=float(np.abs(d).mean()), sd=float(d.std()))
            righe.append(r)
        tn, en = tru['npop'][s], est['npop'][s]
        righe.append(dict(gruppo=nome, metrica='npop', n=int(s.sum()), n_validi=int(s.sum()),
                          vero=float(np.median(tn)), stima=float(np.mean(en)),
                          giusto_pct=float(100 * np.mean(np.round(en) == np.round(tn)))))
    return pd.DataFrame(righe)


def errore_angolare(res, dirs_vere):
    """Errore angolare (gradi) di ogni popolazione stimata rispetto alla fibra vera piu' vicina.
    dirs_vere: (..., 2, 3) con NaN dove la fibra non c'e'. Ritorna (..., 2), NaN se manca una delle due."""
    out = np.full(res.shape[:-1] + (2,), np.nan)
    V = np.nan_to_num(dirs_vere)
    for p in (0, 1):
        e = np.stack([res[..., IX[f'dir{p + 1}_{c}']] for c in 'xyz'], -1).astype(float)
        cos = np.abs(np.einsum('...c,...kc->...k', np.nan_to_num(e), V))        # (..., 2)
        cos[~np.isfinite(dirs_vere[..., 0])] = -1
        best = np.clip(cos.max(-1), 0, 1)
        ok = np.isfinite(e).all(-1) & np.isfinite(dirs_vere[..., 0, 0])
        out[..., p] = np.where(ok, np.degrees(np.arccos(best)), np.nan)
    return out


def versione():
    return dbsi_toolbox.__version__

In [ ]:
"""Figure delle simulazioni di accuratezza (matplotlib, PNG). Colori per ruolo:
- mappe di valore: rampa sequenziale a una tinta (blu), chiaro = 0;
- mappe di errore: divergente blu <-> rosso con grigio neutro a zero, scala simmetrica;
- categorie (SNR, bracci, protocolli): slot categoriali in ordine fisso, mai ciclati.
"""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, ListedColormap, BoundaryNorm


# palette di riferimento (validata): slot categoriali, rampa blu, divergente blu-rosso
CAT = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#4a3aa7', '#e34948']
INK, INK2, GRID = '#0b0b0b', '#52514e', '#e4e3df'
SEQ = LinearSegmentedColormap.from_list('seq_blu', ['#f7f9fc', '#cde2fb', '#86b6ef', '#3987e5',
                                                    '#256abf', '#184f95', '#0d366b'])
DIV = LinearSegmentedColormap.from_list('div_br', ['#104281', '#3987e5', '#9ec5f4', '#f0efec',
                                                   '#f4a3a2', '#e34948', '#a12a2a'])
SEQ.set_bad('#ffffff'); DIV.set_bad('#ffffff')
NPOP_CMAP = ListedColormap(['#f0efec', CAT[0], CAT[1]])
NPOP_NORM = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], 3)

ETICHETTA = dict(FF='FF (fibra)', RF='RF (restricted)', HF='HF (hindered)', WF='WF (acqua libera)',
                 ADw='AD pesata', RDw='RD pesata', FAw='FA pesata', npop='n. popolazioni')
INTERVALLO = dict(FF=(0, 0.8), RF=(0, 0.6), HF=(0, 0.9), WF=(0, 1.0), ADw=(1.0, 2.2), RDw=(0.1, 1.0),
                  FAw=(0, 1.0))
ERR_MAX = dict(FF=0.25, RF=0.25, HF=0.25, WF=0.25, ADw=0.6, RDw=0.4, FAw=0.3)


def stile():
    plt.rcParams.update({
        'figure.dpi': 110, 'savefig.dpi': 160, 'savefig.bbox': 'tight', 'font.size': 9,
        'axes.edgecolor': INK2, 'axes.labelcolor': INK, 'axes.titlesize': 10, 'axes.titleweight': 'bold',
        'xtick.color': INK2, 'ytick.color': INK2, 'axes.spines.top': False, 'axes.spines.right': False,
        'axes.grid': False, 'grid.color': GRID, 'grid.linewidth': 0.6, 'legend.frameon': False,
        'figure.facecolor': 'white', 'axes.facecolor': 'white'})


def _lab(k):
    u = UNITA.get(k, '')
    return ETICHETTA.get(k, k) + (f' [{u}]' if u else '')


# ─────────────────────────────────────────────────────────────────────────────
# immagini: vero / stimato / errore
# ─────────────────────────────────────────────────────────────────────────────
def mappe_vero_stima_errore(tru, est, z=None, metriche=None, titolo='', file=None):
    """Una riga per metrica: verita', stima, errore (stima - vero). Le diffusivita' in e-3 mm^2/s.
    Il tensore e' bianco dove non c'e' fibra (vera o stimata)."""
    metriche = list(metriche or (*METRICHE, 'npop'))
    z = tru['FF'].shape[2] // 2 if z is None else z
    fig, axs = plt.subplots(len(metriche), 3, figsize=(8.4, 2.45 * len(metriche)), constrained_layout=True)
    for r, k in enumerate(metriche):
        t = tru[k][:, :, z].T
        e = est[k][:, :, z].T
        if k == 'npop':
            for c, (img, nome) in enumerate(((t, 'vero'), (e, 'stimato'))):
                im = axs[r, c].imshow(img, origin='lower', cmap=NPOP_CMAP, norm=NPOP_NORM, interpolation='nearest')
                axs[r, c].set_title(f'{ETICHETTA[k]} — {nome}')
            cb = fig.colorbar(im, ax=axs[r, :2], ticks=[0, 1, 2], shrink=0.85)
            giusto = np.round(e) == np.round(t)
            axs[r, 2].imshow(np.where(giusto, 1.0, 0.0), origin='lower', cmap=ListedColormap(['#e34948', '#f0efec']),
                             vmin=0, vmax=1, interpolation='nearest')
            axs[r, 2].set_title(f'conteggio giusto: {100 * giusto.mean():.0f}% (rosso = sbagliato)')
        else:
            s = SCALA[k]
            lo, hi = INTERVALLO[k]
            for c, (img, nome) in enumerate(((t, 'vero'), (e, 'stimato'))):
                im = axs[r, c].imshow(img * s, origin='lower', cmap=SEQ, vmin=lo, vmax=hi, interpolation='nearest')
                axs[r, c].set_title(f'{ETICHETTA[k]} — {nome}')
            fig.colorbar(im, ax=axs[r, :2], shrink=0.85, label=UNITA[k] or None)
            d = (e - t) * s
            em = ERR_MAX[k]
            im2 = axs[r, 2].imshow(d, origin='lower', cmap=DIV, vmin=-em, vmax=em, interpolation='nearest')
            tit = f'errore: bias {np.nanmean(d):+.3f}, MAE {np.nanmean(np.abs(d)):.3f}'
            if k in TENSORE:
                # fibra vera ma nessun tensore stimato (nero) / tensore stimato senza fibra vera (rosa)
                mancata = np.isfinite(t) & ~np.isfinite(e)
                falsa = ~np.isfinite(t) & np.isfinite(e)
                ov = np.zeros(t.shape + (4,))
                ov[mancata] = (0.10, 0.10, 0.10, 1.0)
                ov[falsa] = (0.91, 0.48, 0.64, 1.0)
                axs[r, 2].imshow(ov, origin='lower', interpolation='nearest')
                nv = max(np.isfinite(t).sum(), 1)
                tit += f'\nmancate (nero) {100 * mancata.sum() / nv:.0f}%, false (rosa) {falsa.sum()} vox'
            axs[r, 2].set_title(tit, fontsize=9)
            fig.colorbar(im2, ax=axs[r, 2], shrink=0.85)
        for c in range(3):
            axs[r, c].set_xticks([]); axs[r, c].set_yticks([])
    if titolo:
        fig.suptitle(titolo, fontsize=11, fontweight='bold')
    if file:
        fig.savefig(file)
    return fig


def mappa_regioni(pesi, z=None, file=None):
    """Le regioni del fantoccio anatomico: il tessuto dominante in ogni voxel."""
    nomi = list(pesi)
    W = np.stack([pesi[n] for n in nomi], -1)
    z = W.shape[2] // 2 if z is None else z
    dom = W[:, :, z].argmax(-1).T
    pal = ['#d9d7d0', CAT[0], CAT[2], CAT[6], '#9ec5f4', CAT[1], CAT[3], CAT[7], CAT[4]]
    fig, ax = plt.subplots(figsize=(4.6, 4.2), constrained_layout=True)
    ax.imshow(dom, origin='lower', cmap=ListedColormap(pal[:len(nomi)]), vmin=-0.5, vmax=len(nomi) - 0.5,
              interpolation='nearest')
    for q, n in enumerate(nomi):
        ax.plot([], [], 's', color=pal[q], label=n, markersize=8)
    ax.legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title('Fantoccio anatomico: tessuto dominante')
    if file:
        fig.savefig(file)
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# batteria: errore per scenario, per SNR o per braccio
# ─────────────────────────────────────────────────────────────────────────────
def errori_per_scenario(voxel, metriche, gruppo_col='SNR', titolo='', file=None, ordine=None):
    """voxel: DataFrame con colonne scenario, metrica, errore (stima - vero, gia' scalato), e gruppo_col.
    Punto = errore mediano, barra = 10-90 percentile dei voxel; una colonna per metrica."""
    gruppi = list(dict.fromkeys(voxel[gruppo_col])) if ordine is None else ordine
    scen = list(dict.fromkeys(voxel['scenario']))
    n = len(metriche)
    fig, axs = plt.subplots(1, n, figsize=(2.35 * n + 1.6, 0.36 * len(scen) + 1.4), sharey=True,
                            constrained_layout=True)
    axs = np.atleast_1d(axs)
    off = np.linspace(-0.25, 0.25, len(gruppi)) if len(gruppi) > 1 else [0.0]
    for a, k in zip(axs, metriche):
        a.axvline(0, color=INK2, lw=0.8)
        a.grid(axis='x')
        sub = voxel[voxel.metrica == k]
        for gi, g in enumerate(gruppi):
            s = sub[sub[gruppo_col] == g]
            for si, sc in enumerate(scen):
                e = s.loc[s.scenario == sc, 'errore'].to_numpy()
                e = e[np.isfinite(e)]
                if e.size < 3:
                    continue
                y = si + off[gi]
                lo, med, hi = np.percentile(e, [10, 50, 90])
                a.plot([lo, hi], [y, y], color=CAT[gi], lw=2, solid_capstyle='round')
                a.plot(med, y, 'o', color=CAT[gi], ms=5, mec='white', mew=1)
        a.set_title(_lab(k), fontsize=9)
        a.set_xlabel('stima − vero')
    axs[0].set_yticks(range(len(scen)))
    axs[0].set_yticklabels(scen)
    axs[0].invert_yaxis()
    for gi, g in enumerate(gruppi):
        axs[-1].plot([], [], 'o-', color=CAT[gi], label=f'{gruppo_col} {g}' if gruppo_col == 'SNR' else str(g))
    axs[-1].legend(loc='upper left', bbox_to_anchor=(1.02, 1))
    if titolo:
        fig.suptitle(titolo, fontsize=11, fontweight='bold')
    if file:
        fig.savefig(file)
    return fig


def conteggio_popolazioni(voxel_npop, gruppo_col='SNR', file=None, titolo=''):
    """Quota di voxel con 0 / 1 / 2 popolazioni stimate, per scenario (barra piena = verita')."""
    gruppi = list(dict.fromkeys(voxel_npop[gruppo_col]))
    scen = list(dict.fromkeys(voxel_npop['scenario']))
    fig, axs = plt.subplots(1, len(gruppi), figsize=(3.3 * len(gruppi) + 1.4, 0.36 * len(scen) + 1.4),
                            sharey=True, constrained_layout=True)
    axs = np.atleast_1d(axs)
    col = ['#c9c7c0', CAT[0], CAT[1]]
    for a, g in zip(axs, gruppi):
        s = voxel_npop[voxel_npop[gruppo_col] == g]
        for si, sc in enumerate(scen):
            v = s[s.scenario == sc]
            q = [np.mean(np.round(v.stima) == p) for p in (0, 1, 2)]
            left = 0
            for p in range(3):
                a.barh(si, q[p], left=left, color=col[p], height=0.7, edgecolor='white', linewidth=1)
                left += q[p]
            vero = int(np.round(v.vero.median()))
            a.text(1.02, si, f'vero {vero}', va='center', fontsize=7.5, color=INK2)
        a.set_xlim(0, 1.25); a.set_xticks([0, 0.5, 1]); a.set_xticklabels(['0', '50%', '100%'])
        a.set_title(f'{gruppo_col} {g}' if gruppo_col == 'SNR' else str(g), fontsize=9)
    axs[0].set_yticks(range(len(scen))); axs[0].set_yticklabels(scen); axs[0].invert_yaxis()
    from matplotlib.patches import Patch
    axs[-1].legend(handles=[Patch(color=col[p], label=f'{p} popolazioni') for p in range(3)],
                   loc='upper left', bbox_to_anchor=(1.0, 1))
    if titolo:
        fig.suptitle(titolo, fontsize=11, fontweight='bold')
    if file:
        fig.savefig(file)
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# campi continui: stima contro verita'
# ─────────────────────────────────────────────────────────────────────────────
def stima_contro_vero(tru, est, metriche=None, titolo='', file=None):
    metriche = list(metriche or METRICHE)
    n = len(metriche)
    nc = 4
    nr = int(np.ceil(n / nc))
    fig, axs = plt.subplots(nr, nc, figsize=(3.0 * nc, 3.0 * nr), constrained_layout=True)
    axs = np.atleast_2d(axs).ravel()
    for a, k in zip(axs, metriche):
        t = tru[k].ravel() * SCALA[k]
        e = est[k].ravel() * SCALA[k]
        ok = np.isfinite(t) & np.isfinite(e)
        lo, hi = INTERVALLO[k]
        a.hexbin(t[ok], e[ok], gridsize=40, extent=(lo, hi, lo, hi), cmap=SEQ, mincnt=1, bins='log',
                 linewidths=0)
        a.plot([lo, hi], [lo, hi], color=INK, lw=1, ls='--')
        a.set_xlim(lo, hi); a.set_ylim(lo, hi); a.set_aspect('equal')
        d = e[ok] - t[ok]
        r = np.corrcoef(t[ok], e[ok])[0, 1] if ok.sum() > 2 else np.nan
        mancanti = np.mean(~np.isfinite(e[np.isfinite(t)])) if k in TENSORE else 0
        txt = f'r {r:.2f}\nbias {d.mean():+.3f}\nMAE {np.abs(d).mean():.3f}'
        if k in TENSORE:
            txt += f'\nfibra mancata {100 * mancanti:.0f}%'
        a.text(0.04, 0.96, txt, transform=a.transAxes, va='top', fontsize=8, color=INK)
        a.set_title(_lab(k), fontsize=9)
        a.set_xlabel('vero'); a.set_ylabel('stimato')
    for a in axs[n:]:
        a.axis('off')
    if titolo:
        fig.suptitle(titolo, fontsize=11, fontweight='bold')
    if file:
        fig.savefig(file)
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# incertezza: z = (stima - vero) / SE
# ─────────────────────────────────────────────────────────────────────────────
def istogrammi_z(zdf, metriche, gruppo_col='braccio', titolo='', file=None):
    """zdf: colonne metrica, z, gruppo_col. Curva grigia = N(0,1): se la SE descrive l'errore, gli
    istogrammi la seguono; piu' stretti = SE conservativa, spostati = bias."""
    gruppi = list(dict.fromkeys(zdf[gruppo_col]))
    nc = 4
    nr = int(np.ceil(len(metriche) / nc))
    fig, axs = plt.subplots(nr, nc, figsize=(3.1 * nc + 1.2, 2.8 * nr), sharey=True, constrained_layout=True)
    axs = np.atleast_1d(axs).ravel()
    for a in axs[len(metriche):]:
        a.axis('off')
    xs = np.linspace(-5, 5, 200)
    for a, k in zip(axs, metriche):
        a.plot(xs, np.exp(-xs ** 2 / 2) / np.sqrt(2 * np.pi), color='#9a988f', lw=2)
        for gi, g in enumerate(gruppi):
            z = zdf[(zdf.metrica == k) & (zdf[gruppo_col] == g)].z.to_numpy()
            z = z[np.isfinite(z)]
            if z.size < 10:
                continue
            a.hist(np.clip(z, -5, 5), bins=np.linspace(-5, 5, 41), density=True, histtype='step', lw=2,
                   color=CAT[gi])
            cop = np.mean(np.abs(z) <= 1.96)
            a.text(0.03, 0.95 - 0.1 * gi, f'cop. 95%: {100 * cop:.0f}%', transform=a.transAxes,
                   fontsize=7.5, color=INK, va='top')
            a.plot([], [], color=CAT[gi], lw=2, label=str(g))
        a.set_title(_lab(k), fontsize=9)
        a.set_xlabel('z = (stima − vero) / SE')
    axs[len(metriche) - 1].legend(loc='upper left', bbox_to_anchor=(1.02, 1))
    if titolo:
        fig.suptitle(titolo, fontsize=11, fontweight='bold')
    if file:
        fig.savefig(file)
    return fig


# ─────────────────────────────────────────────────────────────────────────────
# riepilogo per protocollo / per SNR
# ─────────────────────────────────────────────────────────────────────────────
def mae_per_gruppo(tab, metriche, gruppo_col, titolo='', file=None):
    """tab: colonne metrica, mae, gruppo_col (una riga per scenario). Barre = MAE medio sugli scenari."""
    gruppi = list(dict.fromkeys(tab[gruppo_col]))
    fig, axs = plt.subplots(1, len(metriche), figsize=(2.2 * len(metriche) + 1.0, 2.6), constrained_layout=True)
    axs = np.atleast_1d(axs)
    for a, k in zip(axs, metriche):
        v = [tab[(tab.metrica == k) & (tab[gruppo_col] == g)].mae.mean() for g in gruppi]
        a.bar(range(len(gruppi)), v, color=[CAT[i] for i in range(len(gruppi))], width=0.7)
        for i, x in enumerate(v):
            a.text(i, x, f'{x:.3f}', ha='center', va='bottom', fontsize=7.5, color=INK)
        a.set_xticks(range(len(gruppi))); a.set_xticklabels([str(g) for g in gruppi], fontsize=8)
        a.set_title(_lab(k), fontsize=9)
        a.grid(axis='y')
    axs[0].set_ylabel('MAE medio sugli scenari')
    if titolo:
        fig.suptitle(titolo, fontsize=11, fontweight='bold')
    if file:
        fig.savefig(file)
    return fig

In [ ]:
if RAPIDO:
    N_CAL = 200                     # prova veloce: calibrazione su meno voxel
import json, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings('ignore', category=RuntimeWarning)
stile()

VER = versione()
_vt = tuple(int(x) for x in VER.split('.')[:3])
assert _vt >= (1, 8, 1), f'serve pyDBSI >= 1.8.1, trovata {VER}'
DN = trova_dwidenoise(DWIDENOISE)
print(f'pyDBSI {VER}\ndwidenoise: {DN or "NON TROVATO -> solo il braccio grezzo"}')
if DN is None:
    BRACCI = ['grezzo']

P3_BVAL = """0 0 0 0 0 0 0 0 0 300 300 300 700 700 700 700 700 700 700 700 700 700 700 700 1000 1000 1000 1000 1000 1000 1000 1000 1000 1000 1000 1000 1000 1000 1000 1000 1000 1000 1000 1000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000 2000"""
P3_BVEC = """0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 -0.827892 -0.443607 0.388907 -0.195963 -0.021996 -0.772169 0.428901 -0.482508 -0.400001 0.617707 -0.971440 0.439716 0.074098 -0.770731 0.947588 0.608595 0.842131 0.215671 0.820813 0.526317 0.116272 0.998930 -0.041864 0.466815 0.706768 -0.330802 0.873779 0.029363 -0.077901 0.845481 -0.543135 0.465775 0.265880 -0.577315 0.758291 0.436608 0.283505 0.704573 0.140756 0.476272 0.609103 -0.048362 0.806042 0.180569 0.162519 0.848290 -0.221219 0.641808 0.439881 -0.207121 0.956201 -0.155265 0.268266 0.760663 -0.455737 0.828128 0.129733 -0.184321 0.974578 -0.468655 0.464696 0.519300 -0.550940 0.956542 -0.229050 -0.033559 0.850511 -0.695282 0.671407 0.177585 -0.506768 0.976012 -0.562832 0.187547 0.596676 -0.797793 0.821416 -0.202643 -0.348672 0.863482 -0.812656 0.414584 0.250297
0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 -0.533708 0.295357 -0.913022 -0.734983 -0.999750 -0.459459 -0.661673 -0.764722 -0.068022 -0.777409 -0.189738 -0.054967 -0.625809 0.349569 -0.263092 -0.166643 -0.426494 -0.509055 0.201112 -0.825043 0.029624 -0.046251 -0.844528 0.547826 -0.657461 -0.255804 0.485365 -0.999562 0.525752 -0.158235 -0.648146 0.848154 -0.817423 0.198014 0.438381 -0.787298 -0.624617 -0.578781 -0.895757 -0.353697 -0.785226 -0.690718 -0.279942 -0.978898 -0.296732 -0.525955 -0.865470 -0.043836 -0.880729 -0.431172 -0.172858 -0.986372 0.023712 -0.595628 -0.665170 0.135875 -0.942836 -0.102248 -0.195813 -0.870073 0.285828 -0.696926 -0.362969 0.195848 -0.926285 0.226112 -0.294319 -0.641060 0.454250 -0.769284 -0.011808 0.154341 -0.806667 0.500933 -0.414886 -0.326046 0.516070 -0.767736 0.334839 0.046067 -0.582729 0.686266 -0.505591
0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 0.000000 -0.172482 0.846154 -0.123058 0.649153 -0.003912 0.438922 0.615006 -0.427066 0.913987 -0.118627 -0.142488 0.896453 -0.776448 0.532705 0.181270 0.775785 0.330026 0.833276 0.534621 0.205656 0.992776 -0.000113 0.533872 0.694241 -0.261198 0.908369 0.030521 0.003816 0.847063 -0.510024 0.533771 0.252367 -0.511006 0.792147 -0.482511 0.435357 0.727653 0.410596 0.421672 0.805024 0.111416 0.721505 0.521467 0.095675 0.941030 0.061435 0.449471 0.765611 -0.175560 0.878175 0.236219 0.054427 0.963053 -0.258108 0.591484 0.543822 -0.306969 0.977533 -0.108878 0.152761 0.838069 -0.494592 0.751478 0.216037 -0.299219 0.973523 -0.435899 0.325000 0.585550 -0.613731 0.862001 -0.153558 -0.180301 0.844922 -0.686912 0.507169 0.242793 -0.607879 0.875392 -0.502271 0.004084 0.597628 -0.825669"""
def _da_testo(bval_txt, bvec_txt):
    import io as _io
    bv = np.loadtxt(_io.StringIO(bval_txt)).ravel()
    bc = np.loadtxt(_io.StringIO(bvec_txt)); bc = bc.T if bc.shape[0] == 3 else bc
    n = np.linalg.norm(bc, axis=1, keepdims=True); n[n == 0] = 1
    return bv, bc / n

PROTOCOLLI = {'P3': _da_testo(P3_BVAL, P3_BVEC)}
for P in (1, 2, 4, 5):
    base = Path(ROOT_5P) / f'{P:06d}' / SESSIONE / 'DWI_4s' / 'prep'
    fb, fv = base / f'{STEM}_corrected.bval', base / f'{STEM}_corrected.bvec'
    if fb.is_file() and fv.is_file():
        PROTOCOLLI[f'P{P}'] = carica_protocollo(fb, fv)
for k, p in PROTOCOLLI.items():
    print(f'  {k}: {len(p[0])} volumi — {descrivi_protocollo(p)}')

OUT = Path.cwd() / 'risultati_simulazioni' / f'v{VER.replace(".", "")}' / ('rapido' if RAPIDO else 'completo')
FIG = OUT / 'figure'; FIG.mkdir(parents=True, exist_ok=True)
print('risultati in', OUT)

## 2 — Le condizioni

Per ogni condizione: volume pulito → rumore → (dwidenoise) → fit con calibrazione libera. Si salvano le
mappe stimate, le incertezze e la calibrazione scelta. Il volume pulito dipende solo dal protocollo (stesso
seme per tutti): le condizioni differiscono **solo** per rumore e catena, quindi si confrontano direttamente.

In [ ]:
CONDIZIONI = [('P3', s, BRACCI[0]) for s in SNR_LISTA]
CONDIZIONI += [('P3', s, b) for b in BRACCI[1:] for s in SNR_PRODUZIONE]
CONDIZIONI += [(p, SNR_RIF, BRACCI[0]) for p in PROTOCOLLI if p != 'P3']

VOLUMI = {p: volume_composito(prot, d_variabile=D_VARIABILE, piccolo=RAPIDO) for p, prot in PROTOCOLLI.items()}
A0, MASK0, PARTI0 = VOLUMI['P3']
print(f'volume {A0.shape[:3]}, {MASK0.sum():,} voxel nella maschera; {len(CONDIZIONI)} condizioni')

def condizione(p, snr, braccio):
    f = OUT / f'{p}_snr{snr}_{braccio}.npz'
    if f.is_file() and not RIFARE:
        z = np.load(f, allow_pickle=True)
        return z['res'], z['se'], json.loads(str(z['info']))
    A, mask, _ = VOLUMI[p]
    t0 = time.time()
    res, m, info = esegui(A, PROTOCOLLI[p], snr, braccio, dwidenoise=DN, seed=1000 + snr, mask=mask,
                          n_calibration_voxels=N_CAL)
    info.update(protocollo=p, secondi=round(time.time() - t0), versione=VER, d_variabile=D_VARIABILE,
                rapido=RAPIDO)
    se = m.uncertainty_['se'].astype(np.float32)
    np.savez_compressed(f, res=res.astype(np.float32), se=se, info=json.dumps(info))
    return res, se, info

RIS, INFO = {}, []
t_tot = time.time()
for i, (p, snr, b) in enumerate(CONDIZIONI, 1):
    t0 = time.time()
    res, se, info = condizione(p, snr, b)
    RIS[(p, snr, b)] = (res, se)
    INFO.append(info)
    print(f'[{i}/{len(CONDIZIONI)}] {p} SNR {snr:>2} {b:<10}  lambda_aniso {info["lambda_aniso"]:.4g}  '
          f'lambda_iso {info["lambda_iso"]:.4g}  sigma mappa/vera {info["sigma_mappa_mediana"] / info["sigma_vera"]:.3f}  '
          f'({time.time() - t0:.0f} s; totale {(time.time() - t_tot) / 60:.0f} min)', flush=True)
CAL = pd.DataFrame(INFO)[['protocollo', 'snr', 'braccio', 'lambda_aniso', 'lambda_iso', 'n_iso',
                          'sigma_vera', 'sigma_mappa_mediana', 'secondi']]
CAL.to_csv(OUT / 'calibrazioni.csv', index=False)
display(CAL)

def mappe(p, snr, b, parte):
    """Mappe stimate (nome breve) e incertezze ritagliate su una parte del volume."""
    res, se = RIS[(p, snr, b)]
    sl = VOLUMI[p][2][parte]['sl']
    out = {k: res[sl][..., IX[c]].astype(float) for k, c in METRICHE.items()}
    out['npop'] = np.nan_to_num(res[sl][..., IX['n_fiber_populations']].astype(float))
    for k, c in METRICHE.items():
        out['SE_' + k] = se[sl][..., IX[c]].astype(float)
    out['_res'] = res[sl]
    return out

## 3 — Il fantoccio anatomico

Il tessuto dominante in ogni voxel e, sotto, le mappe vere. Dove due fasci si sovrappongono nasce un
incrocio (A×B a 90°, A×C a 60°); ai bordi c'è volume parziale. Il tensore vero è definito dove la fibra
pesa almeno il 5% del voxel.

In [ ]:
AN = PARTI0['anatomico']
mappa_regioni(AN['pesi'], file=FIG / '01_fantoccio_regioni.png'); plt.show()

## 4 — Vero, stimato, errore (anatomico, catena di produzione)

Una riga per mappa: verità, stima, errore = stima − vero (rosso = sovrastima, blu = sottostima, scala
simmetrica). Nel tensore: **nero** = fibra vera senza tensore stimato (fibra mancata), **rosa** = tensore
stimato dove non c'è fibra vera (fibra falsa). Il conteggio delle popolazioni: rosso dove è sbagliato.

In [ ]:
E = mappe('P3', SNR_RIF, BRACCI[0], 'anatomico')
mappe_vero_stima_errore(AN['verita'], E, titolo=f'Anatomico — P3, SNR {SNR_RIF}, {BRACCI[0]}',
                        file=FIG / f'02_anatomico_P3_snr{SNR_RIF}_{BRACCI[0]}.png'); plt.show()

In [ ]:
# Per regione (solo i voxel "puri": il tessuto dominante pesa almeno il 90%)
W = np.stack([AN['pesi'][t] for t in AN['pesi']], -1)
puri = W.max(-1) >= 0.9
reg = np.where(puri, W.argmax(-1), -1)
righe = []
for (p, snr, b) in CONDIZIONI:
    if p != 'P3':
        continue
    T = tabella_errori(mappe('P3', snr, b, 'anatomico'), AN['verita'], reg, list(AN['pesi']))
    T['SNR'], T['braccio'] = snr, b
    righe.append(T)
TAB_AN = pd.concat(righe, ignore_index=True)
TAB_AN.to_csv(OUT / 'anatomico_per_regione.csv', index=False)
v = TAB_AN[(TAB_AN.SNR == SNR_RIF) & (TAB_AN.braccio == BRACCI[0]) & TAB_AN.metrica.isin(list(METRICHE))]
print(f'bias per regione (P3, SNR {SNR_RIF}, {BRACCI[0]}; diffusivita in e-3 mm^2/s):')
display(v.pivot(index='gruppo', columns='metrica', values='bias')[list(METRICHE)].round(3))

In [ ]:
if ('P3', SNR_RIF, 'produzione') in RIS:
    Ep = mappe('P3', SNR_RIF, 'produzione', 'anatomico')
    mappe_vero_stima_errore(AN['verita'], Ep, metriche=list(METRICHE),
                            titolo=f'Anatomico — P3, SNR {SNR_RIF}, con dwidenoise (sensibilita, fantoccio idealizzato)',
                            file=FIG / f'03_anatomico_P3_snr{SNR_RIF}_produzione.png'); plt.show()

## 5 — Batteria: errore per scenario

Per ogni scenario e mappa: punto = errore mediano dei voxel, barra = 10°–90° percentile (quanto varia da
voxel a voxel). Lo zero è la verità. Solo i voxel interni ai blocchi (al bordo dwidenoise mescola due
scenari). Le diffusività in 1e-3 mm²/s.

**Come leggere i numeri.** Il *bias* è l'errore medio: dice se lo scenario è stimato sistematicamente troppo
alto o troppo basso. Il *MAE* è la media degli errori **voxel per voxel** in valore assoluto: contiene anche la
dispersione dovuta al rumore, quindi è sempre più grande del bias. (Il riferimento di luglio, MAE FF 0.030, era
l'errore della **media** di ogni scenario: è il bias di qui, non il MAE.) Il tensore dei voxel con poca fibra
(GM, edema, fibra debole) è poco determinato dal dato: lì l'errore del tensore è grande per ragioni fisiche,
non per un difetto del fit (vedi le mappe di incertezza).

In [ ]:
BA = PARTI0['batteria']
INN = interno(BA['etichette'])
NOMI_SC = BA['nomi']

def voxel_batteria(p, snr, b):
    """DataFrame lungo: un errore per voxel, scenario e metrica (piu' npop e SE per l'incertezza)."""
    E = mappe(p, snr, b, 'batteria'); tru = VOLUMI[p][2]['batteria']['verita']
    lab = VOLUMI[p][2]['batteria']['etichette']; inn = interno(lab)
    out = []
    for k in METRICHE:
        t, e, se = tru[k][inn], E[k][inn], E['SE_' + k][inn]
        out.append(pd.DataFrame(dict(scenario=np.array(NOMI_SC)[lab[inn]], metrica=k,
                                     vero=t * SCALA[k], stima=e * SCALA[k], errore=(e - t) * SCALA[k],
                                     z=(e - t) / se)))
    out.append(pd.DataFrame(dict(scenario=np.array(NOMI_SC)[lab[inn]], metrica='npop', vero=tru['npop'][inn],
                                 stima=E['npop'][inn], errore=E['npop'][inn] - tru['npop'][inn], z=np.nan)))
    d = pd.concat(out, ignore_index=True); d['protocollo'], d['SNR'], d['braccio'] = p, snr, b
    return d

VB = pd.concat([voxel_batteria(*c) for c in CONDIZIONI], ignore_index=True)
VB.to_csv(OUT / 'batteria_voxel.csv.gz', index=False)

sel = VB[(VB.protocollo == 'P3') & (VB.braccio == BRACCI[0])]
errori_per_scenario(sel, FRAZIONI, gruppo_col='SNR', titolo=f'Batteria — frazioni (P3, {BRACCI[0]})',
                    file=FIG / f'04_batteria_frazioni_P3_{BRACCI[0]}.png'); plt.show()
errori_per_scenario(sel, TENSORE, gruppo_col='SNR', titolo=f'Batteria — tensore pesato (P3, {BRACCI[0]})',
                    file=FIG / f'05_batteria_tensore_P3_{BRACCI[0]}.png'); plt.show()

In [ ]:
conteggio_popolazioni(sel[sel.metrica == 'npop'], gruppo_col='SNR',
                      titolo=f'Batteria — popolazioni stimate (P3, {BRACCI[0]})',
                      file=FIG / f'06_batteria_popolazioni_P3_{BRACCI[0]}.png'); plt.show()

In [ ]:
# Tabella: per scenario, a SNR_RIF, vero / stima mediana / bias / MAE; e il riepilogo per SNR
def riassunto(d):
    g = d[d.metrica != 'npop'].groupby(['scenario', 'metrica'], sort=False)
    return pd.DataFrame(dict(vero=g.vero.median(), stima=g.stima.median(), bias=g.errore.mean(),
                             mae=g.errore.apply(lambda x: np.nanmean(np.abs(x))),
                             mancanti_pct=g.stima.apply(lambda x: 100 * np.mean(~np.isfinite(x))))).reset_index()
R = riassunto(sel[sel.SNR == SNR_RIF])
R.to_csv(OUT / f'batteria_P3_snr{SNR_RIF}_{BRACCI[0]}.csv', index=False)
print(f'bias per scenario (P3, SNR {SNR_RIF}, {BRACCI[0]}):')
display(R.pivot(index='scenario', columns='metrica', values='bias').reindex(NOMI_SC)[list(METRICHE)].round(3))
M = pd.concat([riassunto(sel[sel.SNR == s]).assign(SNR=s) for s in SNR_LISTA])
print('MAE medio sui 13 scenari, per SNR:')
display(M.groupby(['SNR', 'metrica']).mae.mean().unstack()[list(METRICHE)].round(3))

In [ ]:
# Errore angolare della direzione principale, per scenario con fibra (gradi)
righe = []
for snr in SNR_LISTA:
    E = mappe('P3', snr, BRACCI[0], 'batteria')
    ang = errore_angolare(E['_res'], BA['verita']['dirs'])
    for s, n in enumerate(NOMI_SC):
        q = ang[..., 0][INN & (BA['etichette'] == s)]
        if np.isfinite(q).any():
            righe.append(dict(scenario=n, SNR=snr, mediana=np.nanmedian(q), p90=np.nanpercentile(q, 90)))
ANG = pd.DataFrame(righe)
ANG.to_csv(OUT / 'batteria_errore_angolare.csv', index=False)
display(ANG.pivot(index='scenario', columns='SNR', values='mediana').round(1))

## 6 — Effetto del denoising

Stessa batteria, stesso rumore, a SNR_RIF: con `dwidenoise` + sua mappa (produzione) contro dati grezzi +
σ vera. Se i due bracci coincidono, il denoising non sposta le stime; dove differiscono, è il denoising
(non lo stimatore) a cambiare il numero.

In [ ]:
if sum(('P3', SNR_RIF, b) in RIS for b in BRACCI) > 1:
    sel2 = VB[(VB.protocollo == 'P3') & (VB.SNR == SNR_RIF)]
    errori_per_scenario(sel2, FRAZIONI, gruppo_col='braccio', ordine=BRACCI,
                        titolo=f'Denoising — frazioni (P3, SNR {SNR_RIF})',
                        file=FIG / f'07_denoising_frazioni_P3_snr{SNR_RIF}.png'); plt.show()
    errori_per_scenario(sel2, TENSORE, gruppo_col='braccio', ordine=BRACCI,
                        titolo=f'Denoising — tensore (P3, SNR {SNR_RIF})',
                        file=FIG / f'08_denoising_tensore_P3_snr{SNR_RIF}.png'); plt.show()
    D = pd.concat([riassunto(sel2[sel2.braccio == b]).assign(braccio=b) for b in BRACCI])
    print('MAE medio sui 13 scenari, per braccio:')
    display(D.groupby(['braccio', 'metrica']).mae.mean().unstack()[list(METRICHE)].round(3))

## 7 — Campi continui: stima contro verità

Ogni punto è un voxel (densità in scala logaritmica); la diagonale tratteggiata è la stima perfetta. Per il
tensore, "fibra mancata" = quota dei voxel con fibra vera in cui il toolbox non stima un tensore.

**Il "gradino" della FF.** Sotto una FF vera di ~0.15 la stima o è 0 o parte da ~0.15: è la soglia di fibra
del toolbox (`fiber_threshold`, applicata alla FF finale dalla 1.8.0). Una fibra più debole della soglia viene
tolta (FF = 0, frazione restituita allo spettro isotropo); una fibra falsa che la supera resta. Sono i due
errori opposti di una soglia, ed è qui che si vedono.

In [ ]:
CA = PARTI0['campi']
for b in [b for b in BRACCI if ('P3', SNR_RIF, b) in RIS]:
    E = mappe('P3', SNR_RIF, b, 'campi')
    stima_contro_vero(CA['verita'], E, titolo=f'Campi continui — P3, SNR {SNR_RIF}, {b}',
                      file=FIG / f'09_campi_P3_snr{SNR_RIF}_{b}.png'); plt.show()

## 8 — Le mappe di incertezza descrivono l'errore?

z = (stima − vero) / SE sui voxel della batteria senza incroci (dove lo stimatore è quasi senza bias). Se
la SE è giusta, gli istogrammi seguono la curva grigia N(0,1) e la copertura al 95% è vicina al 95%.
Attese due deviazioni: con dwidenoise il rumore residuo è più piccolo di σ (SE larga → istogramma più
stretto), e dove c'è bias l'istogramma si sposta.

In [ ]:
cross = [n for n in NOMI_SC if n.startswith('Crossing')]
zsel = VB[(VB.protocollo == 'P3') & (VB.SNR == SNR_RIF) & ~VB.scenario.isin(cross)]
istogrammi_z(zsel, list(METRICHE), gruppo_col='braccio', titolo=f'Incertezza — P3, SNR {SNR_RIF}, senza incroci',
             file=FIG / f'10_incertezza_P3_snr{SNR_RIF}.png'); plt.show()

## 9 — Protocolli dei 5P

La stessa batteria su ogni protocollo trovato, a SNR_RIF e con la catena di produzione: a parità di SNR,
quanto pesa il disegno dell'acquisizione (shell, direzioni) sull'accuratezza. (I 5P veri hanno SNR diversi
fra loro: qui è tenuto fisso apposta.)

In [ ]:
if len(PROTOCOLLI) > 1:
    selp = VB[(VB.SNR == SNR_RIF) & (VB.braccio == BRACCI[0])]
    T = pd.concat([riassunto(selp[selp.protocollo == p]).assign(protocollo=p) for p in PROTOCOLLI])
    T.to_csv(OUT / f'protocolli_snr{SNR_RIF}.csv', index=False)
    mae_per_gruppo(T, list(METRICHE), 'protocollo', titolo=f'MAE per protocollo (SNR {SNR_RIF}, {BRACCI[0]})',
                   file=FIG / f'11_protocolli_mae_snr{SNR_RIF}.png'); plt.show()
    errori_per_scenario(selp, FRAZIONI, gruppo_col='protocollo', ordine=list(PROTOCOLLI),
                        titolo=f'Protocolli — frazioni (SNR {SNR_RIF})',
                        file=FIG / f'12_protocolli_frazioni_snr{SNR_RIF}.png'); plt.show()
else:
    print('solo il P3: le tabelle degli altri protocolli non sono state trovate in', ROOT_5P)

## 10 — Riepilogo

In [ ]:
righe = []
for (p, snr, b) in CONDIZIONI:
    d = VB[(VB.protocollo == p) & (VB.SNR == snr) & (VB.braccio == b)]
    r = riassunto(d)
    riga = dict(protocollo=p, SNR=snr, braccio=b)
    for k in METRICHE:
        riga[f'MAE_{k}'] = r[r.metrica == k].mae.mean()
    n = d[d.metrica == 'npop']
    riga['npop_giusto_pct'] = 100 * np.mean(np.round(n.stima) == np.round(n.vero))
    righe.append(riga)
RIEP = pd.DataFrame(righe)
RIEP.to_csv(OUT / 'riepilogo.csv', index=False)
display(RIEP.round(3))
(OUT / 'leggimi.json').write_text(json.dumps(dict(
    versione=VER, rapido=RAPIDO, d_variabile=D_VARIABILE, snr=SNR_LISTA, snr_rif=SNR_RIF, bracci=BRACCI,
    protocolli={k: descrivi_protocollo(v) for k, v in PROTOCOLLI.items()}, dwidenoise=DN), indent=2))
print('figure in', FIG)
for f in sorted(FIG.glob('*.png')):
    print('  ', f.name)